In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

catalog_name = "workspace"
schema_name = "default"

silver_schema = f"{catalog_name}.{schema_name}"
gold_schema = f"{catalog_name}.{schema_name}"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {gold_schema}")

In [0]:
# =========================================
# GOLD 1: PRODUCT FEATURES
# Grain = 1 row per product
# Supports content-based + popularity-based recommenders
# For: cold-start item recommendation
# =========================================

df_sephora = spark.table(f"{silver_schema}.silver_sephora_product_info")
df_cosmetic = spark.table(f"{silver_schema}.silver_skincare_cosmetic_p")

df_product_features = (
    df_sephora.alias("s")
    .join(
        df_cosmetic.select(
            "brand_product_key",
            "is_combination_skin",
            "is_dry_skin",
            "is_normal_skin",
            "is_oily_skin",
            "is_sensitive_skin",
            "supported_skin_types",
            F.col("ingredient_count").alias("cosmetic_ingredient_count"),
            F.col("rating").alias("cosmetic_rating"),
            F.col("product_type_label").alias("cosmetic_product_type_label")
        ).alias("c"),
        on="brand_product_key",
        how="left"
    )
    .select(
        F.col("s.product_id"),
        F.col("s.product_name"),
        F.col("s.brand_name"),
        F.col("s.brand_name_std"),
        F.col("s.product_name_std"),
        F.col("s.brand_product_key"),
        F.col("s.primary_category"),
        F.col("s.secondary_category"),
        F.col("s.tertiary_category"),
        F.col("s.category_path"),
        F.col("s.current_price_usd"),
        F.col("s.rating").alias("catalog_rating"),
        F.col("s.review_count"),
        F.col("s.loves_count"),
        F.col("s.ingredient_count"),
        F.col("s.is_new"),
        F.col("s.is_online_only"),
        F.col("s.out_of_stock"),
        F.col("s.sephora_exclusive"),
        F.col("s.limited_edition"),
        F.col("s.is_skincare_product"),
        F.col("c.is_combination_skin"),
        F.col("c.is_dry_skin"),
        F.col("c.is_normal_skin"),
        F.col("c.is_oily_skin"),
        F.col("c.is_sensitive_skin"),
        F.col("c.supported_skin_types"),
        F.col("c.cosmetic_ingredient_count"),
        F.col("c.cosmetic_rating"),
        F.col("c.cosmetic_product_type_label")
    )
    .withColumn(
        "has_cosmetic_match",
        F.col("supported_skin_types").isNotNull()
    )
    .withColumn(
        "final_ingredient_count",
        F.coalesce(F.col("cosmetic_ingredient_count"), F.col("ingredient_count"))
    )
    .withColumn(
        "price_bucket",
        F.when(F.col("current_price_usd") < 25, "low")
         .when(F.col("current_price_usd") < 60, "mid")
         .when(F.col("current_price_usd") >= 60, "high")
    )
    .withColumn(
        "popularity_score",
        (
            F.coalesce(F.col("catalog_rating"), F.lit(0.0)) * F.lit(0.5) +
            F.log1p(F.coalesce(F.col("review_count"), F.lit(0))) * F.lit(0.3) +
            F.log1p(F.coalesce(F.col("loves_count"), F.lit(0))) * F.lit(0.2)
        )
    )
    .withColumn("gold_processed_ts", F.current_timestamp())
)

df_product_features.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{gold_schema}.gold_product_features")

display(df_product_features.limit(10))

In [0]:
# =========================================
# GOLD 2: PRODUCT INGREDIENT LONG
# Grain = 1 row per product-ingredient
# Supports ingredient similarity / filtering
# =========================================

df_ing = spark.table(f"{silver_schema}.silver_ingredients_exploded")
df_products = spark.table(f"{gold_schema}.gold_product_features").select("product_id", "brand_product_key")

df_product_ingredient_long = (
    df_products
    .join(df_ing, on="brand_product_key", how="left")
    .select(
        "product_id",
        "ingredient_name_raw",
        "ingredient_name_std",
        "source_system"
    )
    .filter(F.col("ingredient_name_std").isNotNull())
    .withColumn("gold_processed_ts", F.current_timestamp())
)

df_product_ingredient_long.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{gold_schema}.gold_product_ingredient_long")

display(df_product_ingredient_long.limit(10))

In [0]:
df_reviews = spark.table(f"{silver_schema}.silver_sephora_reviews")

In [0]:
# =========================================
# GOLD 4: USER-PRODUCT INTERACTIONS
# Grain = 1 row per user-product interaction
# Supports collaborative filtering
# =========================================

df_reviews = spark.table(f"{silver_schema}.silver_sephora_reviews")

df_user_product_interactions = (
    df_reviews
    .filter(F.col("author_id").isNotNull() & F.col("product_id").isNotNull())
    .select(
        F.col("author_id").alias("user_id"),
        "product_id",
        "rating",
        "is_recommended",
        "review_date"
    )
    .withColumn("interaction_type", F.lit("review"))
    .withColumn(
        "interaction_strength",
        F.when(F.col("rating").isNotNull(), F.col("rating"))
         .when(F.col("is_recommended") == True, F.lit(1.0))
         .otherwise(F.lit(1.0))
    )
    .withColumn("gold_processed_ts", F.current_timestamp())
)

df_user_product_interactions.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{gold_schema}.gold_user_product_interactions")

display(df_user_product_interactions.limit(10))

In [0]:
# =========================================
# GOLD 5: FINAL RECOMMENDER DATASET
# =========================================

df_interactions = spark.table(f"{gold_schema}.gold_user_product_interactions").alias("i")
df_products = spark.table(f"{gold_schema}.gold_product_features").alias("p")
df_reviews = spark.table(f"{gold_schema}.gold_product_review_features").alias("r")

df_recommender = (
    df_interactions
    .join(df_products, on="product_id", how="left")
    .join(df_reviews, on="product_id", how="left")
    .select(
        # Core MF columns
        F.col("i.user_id"),
        F.col("i.product_id"),
        F.col("i.rating"),
        F.col("i.is_recommended"),
        F.col("i.review_date"),
        F.col("i.interaction_type"),
        F.col("i.interaction_strength"),

        # Product features
        F.col("p.product_name"),
        F.col("p.brand_name"),
        F.col("p.primary_category"),
        F.col("p.secondary_category"),
        F.col("p.tertiary_category"),
        F.col("p.category_path"),
        F.col("p.current_price_usd"),
        F.col("p.price_bucket"),
        F.col("p.catalog_rating"),
        F.col("p.review_count").alias("catalog_review_count"),
        F.col("p.loves_count"),
        F.col("p.popularity_score"),

        # Review aggregate features
        F.col("r.avg_review_rating"),
        F.col("r.review_rating_count"),
        F.col("r.recommendation_ratio"),
        F.col("r.avg_review_text_length"),
        F.col("r.avg_helpfulness"),
        F.col("r.latest_review_date"),
        F.col("r.first_review_date")
    )
    .withColumn("gold_processed_ts", F.current_timestamp())
)

df_recommender.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{gold_schema}.gold_recommender_dataset")

display(df_recommender.limit(10))
print("Gold recommender dataset:", df_recommender.count())

In [0]:
%sql
SHOW TABLES IN workspace.default

In [0]:
%sql
SELECT COUNT(*) FROM workspace.default.gold_product_features

In [0]:
%sql
SELECT COUNT(*) FROM workspace.default.gold_product_review_features

In [0]:
%sql
SELECT COUNT(*) FROM workspace.default.gold_user_product_interactions

In [0]:
%sql
SELECT COUNT(*) FROM workspace.default.gold_recommender_training_dataset

In [0]:
%sql
SELECT has_cosmetic_match, COUNT(*)
FROM workspace.default.gold_product_features
GROUP BY has_cosmetic_match
-- All products currently have no cosmetic_p match, which means the enrichment join is failing and skin-type features are not yet being added to Gold.

In [0]:
%sql
SELECT price_bucket, COUNT(*)
FROM workspace.default.gold_product_features
GROUP BY price_bucket
-- Products are reasonably distributed across low, mid, and high price tiers, so price is a usable feature for recommendation and filtering.

In [0]:
%sql
SELECT COUNT(DISTINCT user_id) AS unique_users,
       COUNT(DISTINCT product_id) AS unique_products,
       COUNT(*) AS total_interactions
FROM workspace.default.gold_user_product_interactions
---- The interaction table is large enough to support collaborative filtering, although it only covers a subset of the full product catalog.

In [0]:
%sql
SELECT product_id, product_name, brand_name, popularity_score
FROM workspace.default.gold_product_features
WHERE out_of_stock = false
ORDER BY popularity_score DESC
LIMIT 20
---- Popularity ranking is working as a baseline recommender, but it currently reflects the full Sephora catalog rather than skincare only.